# 02.2 - Audit leakage cua split CNN

Notebook nay kiem tra 3 nguy co optimistic split trong CNN random split:

1. Cung gene xuat hien o ca train/test.
2. Variant test nam rat gan variant train tren genome.
3. Sequence context ref/alt trung nhau giua train/test.

No khong train model, chi audit split hien tai cua notebook 02.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('/mnt/MyData/Bioinformatics/Project')
META_PATH = PROJECT_ROOT / 'processed/clinvar_training_metadata.parquet'
RANDOM_STATE = 42

META_PATH.exists(), META_PATH

In [ ]:
cols = [
    'GeneSymbol', 'Chromosome', 'PositionVCF',
    'ReferenceAlleleVCF', 'AlternateAlleleVCF',
    'ref_seq', 'alt_seq', 'Y',
]
df = pd.read_parquet(META_PATH, columns=cols)
y = df['Y'].to_numpy(dtype=np.int8)

indices = np.arange(len(y))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=RANDOM_STATE, stratify=y[temp_idx]
)

print('rows:', len(df))
print(f"train: {len(train_idx):,}, labels={dict(zip(*np.unique(y[train_idx], return_counts=True)))}")
print(f"val:   {len(val_idx):,}, labels={dict(zip(*np.unique(y[val_idx], return_counts=True)))}")
print(f"test:  {len(test_idx):,}, labels={dict(zip(*np.unique(y[test_idx], return_counts=True)))}")

## 1. Exact variant duplicate

Neu cung `Chromosome + PositionVCF + REF + ALT` nam o ca train/test thi do la leakage ro rang.

In [ ]:
key_cols = ['Chromosome', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']

def key_set(idx):
    return set(map(tuple, df.iloc[idx][key_cols].astype(str).to_numpy()))

for name, a, b in [
    ('train/val', train_idx, val_idx),
    ('train/test', train_idx, test_idx),
    ('val/test', val_idx, test_idx),
]:
    overlap = len(key_set(a) & key_set(b))
    print(f'{name}: exact variant overlap = {overlap:,}')

## 2. Gene overlap

Random split theo row thuong de cung gene xuat hien o ca train/test. Dieu nay lam diem CNN de dep hon vi model gap nhieu context cua cung gene/vung lien quan trong train.

In [ ]:
def gene_set(idx):
    return set(df.iloc[idx]['GeneSymbol'].fillna('unknown').astype(str)) - {'unknown'}

for name, a, b in [
    ('train/val', train_idx, val_idx),
    ('train/test', train_idx, test_idx),
    ('val/test', val_idx, test_idx),
]:
    ga = gene_set(a)
    gb = gene_set(b)
    overlap = ga & gb
    print(
        f'{name}: genes_a={len(ga):,}, genes_b={len(gb):,}, '
        f'overlap={len(overlap):,}, overlap_pct_b={len(overlap) / max(len(gb), 1):.3f}'
    )

## 3. Genome-near overlap

Voi moi variant val/test, tim variant train gan nhat tren cung chromosome. Neu nhieu test variant cach train chi vai bp, split nay van rat de.

In [ ]:
train_pos_by_chr = {}
for chrom, sub in df.iloc[train_idx].groupby('Chromosome', sort=False):
    train_pos_by_chr[str(chrom)] = np.sort(
        pd.to_numeric(sub['PositionVCF'], errors='coerce')
        .dropna()
        .astype(np.int64)
        .to_numpy()
    )

def nearest_train_distances(idx):
    chunks = []
    for chrom, sub in df.iloc[idx].groupby('Chromosome', sort=False):
        arr = train_pos_by_chr.get(str(chrom))
        if arr is None or len(arr) == 0:
            continue
        pos = pd.to_numeric(sub['PositionVCF'], errors='coerce').dropna().astype(np.int64).to_numpy()
        j = np.searchsorted(arr, pos)
        d = np.full(len(pos), np.iinfo(np.int64).max, dtype=np.int64)
        m = j < len(arr)
        d[m] = np.minimum(d[m], np.abs(arr[j[m]] - pos[m]))
        m = j > 0
        d[m] = np.minimum(d[m], np.abs(arr[j[m] - 1] - pos[m]))
        chunks.append(d)
    return np.concatenate(chunks) if chunks else np.array([], dtype=np.int64)

for split_name, idx in [('val', val_idx), ('test', test_idx)]:
    d = nearest_train_distances(idx)
    print('\n', split_name)
    print('quantiles:', {q: int(np.quantile(d, q)) for q in [0, .01, .05, .10, .25, .50, .75, .90, .95, .99]})
    for t in [1, 10, 50, 100, 201, 500, 1000, 10000]:
        print(f'pct <= {t:5,d} bp: {(d <= t).mean():.4f}')

## 4. Exact sequence overlap

`ref_seq` co the trung nhau nhieu vi cac nearby SNV trong cung window co chung reference context. `alt_seq` trung exact it hon, nhung van nen audit.

In [ ]:
for seq_col in ['ref_seq', 'alt_seq']:
    for name, a, b in [
        ('train/val', train_idx, val_idx),
        ('train/test', train_idx, test_idx),
        ('val/test', val_idx, test_idx),
    ]:
        sa = set(df.iloc[a][seq_col].astype(str))
        sb = set(df.iloc[b][seq_col].astype(str))
        overlap = sa & sb
        print(
            f'{name} {seq_col}: exact_seq_overlap={len(overlap):,}, '
            f'pct_b_unique={len(overlap) / max(len(sb), 1):.4f}'
        )

## Ket luan cach doc

- `exact variant overlap = 0`: khong co leakage do duplicate exact variant.
- Gene overlap cao: random split dang optimistic.
- Nearest train distance rat nho: nhieu test variant nam gan train variant trong cung genome window.
- Neu muon test nghiem tuc hon, can train them CNN voi `gene_group` split hoac chromosome holdout.